# FraudScope — 03 Graph Neural Networks
## Phase 3 · Features graphe (IEEE-CIS) · GCN & GAT (Elliptic)

**Objectifs de ce notebook** :
1. Construire un **graphe de transactions** à partir du dataset IEEE-CIS avec NetworkX
2. Extraire des **features graphe** (degré, centralité, vélocité réseau) et mesurer leur apport sur XGBoost
3. Entraîner un **GCN** (Graph Convolutional Network) sur le dataset Elliptic Bitcoin
4. Entraîner un **GAT** (Graph Attention Network) sur le même dataset
5. Comparer GCN, GAT et le meilleur modèle Phase 2 (`FraudScopeXGB v2`) sur les métriques clés
6. Logger tous les runs dans MLflow (expérience `fraud-detection-paytrack`)

**Prérequis** :
- Avoir complété `01_exploration.ipynb` et `02_mlops.ipynb`
- Dataset Elliptic présent dans `data/elliptic/` (voir `data/get_dataset.md`)
- Serveur MLflow actif : `mlflow server --host 127.0.0.1 --port 8080 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts`

> ℹ️ Ce notebook utilise deux datasets distincts :
> - **IEEE-CIS** pour la partie features graphe + XGBoost (Partie A)
> - **Elliptic Bitcoin Dataset** pour l'entraînement GCN/GAT (Partie B)

## 0. Imports et configuration

In [6]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Graph (Partie A) ──────────────────────────────────────────
import networkx as nx

# ── ML classique ─────────────────────────────────────────────
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    recall_score, f1_score, average_precision_score,
    precision_recall_curve, roc_auc_score
)
import xgboost as xgb

# ── GNN (Partie B) ────────────────────────────────────────────
# Installation PyTorch Geometric :
# La commande varie selon la plateforme. Exemples :
#   CPU-only  : pip install torch-geometric
#   CUDA 11.8 : pip install torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
#   Colab     : pip install torch-geometric  (torch est déjà présent)
# Vérifier la compatibilité : https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.utils import to_networkx

# ── MLflow ────────────────────────────────────────────────────
from dotenv import load_dotenv
import mlflow
import mlflow.pytorch
import mlflow.xgboost
from mlflow.tracking import MlflowClient

# ── Configuration ─────────────────────────────────────────────
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

FIGURES_DIR   = Path('figures')
ARTIFACTS_DIR = Path('artifacts')
FIGURES_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE   = 42
TARGET_IEEE    = 'isFraud'
DATA_DIR       = Path('data')
ELLIPTIC_DIR   = DATA_DIR / 'elliptic'
N_SPLITS_TSS   = 5
GRAPH_SAMPLE_N = 10_000
# Cap sur la taille des cliques lors de la construction du graphe.
# Évite l'explosion combinatoire sur les domaines génériques (ex: gmail.com).
# Un groupe de k cartes produit k*(k-1)/2 arêtes ; avec k=50 → 1225 arêtes max/groupe.
MAX_CLIQUE_SIZE = 50

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── MLflow setup ─────────────────────────────────────────────
load_dotenv()
TRACKING_URI    = os.getenv('MLFLOW_TRACKING_URI',    'http://127.0.0.1:8080')
EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'fraud-detection-paytrack')
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Device PyTorch : {DEVICE}')
print(f'MLflow URI     : {TRACKING_URI}')
print(f'Expérience     : {EXPERIMENT_NAME}')
import torch_geometric
print(f'PyG version    : {torch_geometric.__version__}')
print('Imports OK')

Device PyTorch : cpu
MLflow URI     : http://127.0.0.1:8080
Expérience     : fraud-detection-paytrack
PyG version    : 2.8.0
Imports OK


---
# PARTIE A — Features graphe sur IEEE-CIS + XGBoost

**Idée** : les fraudes ne sont pas des événements isolés — elles forment des clusters dans un réseau de transactions partageant des identifiants (cartes, adresses, e-mails, appareils). En modélisant ces relations comme un graphe, on extrait des signaux de structure réseau que les features tabulaires classiques ne capturent pas.

**Graphe construit** : chaque nœud est un identifiant de carte (`card1`) ; une arête relie deux cartes si elles ont effectué une transaction vers le même marchand (`P_emaildomain`) dans la même fenêtre temporelle (même journée). Les poids d'arête sont le nombre de co-occurrences.

## 1. Chargement IEEE-CIS et features Phase 1

In [ ]:
assert (DATA_DIR / 'train_transaction.csv').exists(), "Données IEEE-CIS manquantes → voir data/get_dataset.md"

# ── Dtypes optimisés pour réduire l'empreinte mémoire ────────────────────────
# Raison : pandas charge les ~339 colonnes numériques en float64 par défaut,
# ce qui requiert 339 × 590 540 × 8 octets ≈ 1.5 GiB — trop sur machines < 8 Go RAM.
# float32 divise ce coût par 2 ; int32 pour TransactionID/TransactionDT/isFraud.
# Les colonnes catégorielles (card*, addr*, email, device…) restent en object (str).
# Aucune perte de précision pour XGBoost ni pour les features graphe.

# Colonnes entières connues
_INT32_COLS = ['TransactionID', 'TransactionDT', 'isFraud',
               'card1', 'card2', 'card3', 'card5',
               'addr1', 'addr2',
               'dist1', 'dist2']

def _make_dtype_map(csv_path: Path, int_cols: list, float_dtype=np.float32) -> dict:
    """Lit uniquement l'en-tête du CSV pour construire la map dtype.
    Toutes les colonnes numériques inférées par leur nom (V*, C*, D*, M*, id_*)
    sont castées en float32 ; les colonnes entières connues en Int32.
    """
    header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    dtype_map = {}
    int_set = set(int_cols)
    for col in header:
        if col in int_set:
            dtype_map[col] = 'Int32'   # nullable integer (gère les NaN)
        elif col.startswith(('V', 'C', 'D', 'M', 'id_')):
            dtype_map[col] = float_dtype
        elif col == 'TransactionAmt':
            dtype_map[col] = float_dtype
    return dtype_map

print('Chargement IEEE-CIS (dtypes optimisés)...')
tx_dtypes = _make_dtype_map(DATA_DIR / 'train_transaction.csv', _INT32_COLS)
id_dtypes = _make_dtype_map(DATA_DIR / 'train_identity.csv',    _INT32_COLS)

train_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv', dtype=tx_dtypes)
train_id = pd.read_csv(DATA_DIR / 'train_identity.csv',    dtype=id_dtypes)
df = train_tx.merge(train_id, on='TransactionID', how='left')
df = df.sort_values('TransactionDT').reset_index(drop=True)
print(f'Dataset : {df.shape[0]:,} transactions, {df.shape[1]} colonnes')

# Libérer la mémoire des DataFrames intermédiaires immédiatement
del train_tx, train_id

df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day']  = df['TransactionDT'] // (3600 * 24)
proxy_cols = [c for c in ['card1','card2','card3','card4','card5','card6','addr1','addr2'] if c in df.columns]
df['customer_proxy'] = df[proxy_cols].astype(str).fillna('NA').agg('_'.join, axis=1)
df['amount_ratio'] = df['TransactionAmt'] / (
    df.groupby('customer_proxy')['TransactionAmt'].transform('median').clip(lower=0.01)
)
df['amount_ratio'] = df['amount_ratio'].replace([np.inf, -np.inf], 1.0).fillna(1.0)
df['tx_rank_in_customer'] = df.groupby('customer_proxy').cumcount()
print('Features temporelles et vélocité OK')

## 2. Construction du graphe de transactions (NetworkX)

On travaille sur un **échantillon de `GRAPH_SAMPLE_N` transactions** pour la construction et la visualisation du graphe.

**Nœuds** : valeurs uniques de `card1` (identifiant de carte)
**Arêtes** : deux cartes sont reliées si elles ont la même `P_emaildomain` le même `day`
**Poids** : nombre de co-occurrences

In [ ]:
sample = df.head(GRAPH_SAMPLE_N).copy()
G = nx.Graph()

for _, row in sample.iterrows():
    node = str(row['card1'])
    if not G.has_node(node):
        G.add_node(node, is_fraud=int(row[TARGET_IEEE]))

grp = sample.groupby(['P_emaildomain', 'day'])['card1'].apply(list)
for cards in grp:
    cards = [str(c) for c in cards if pd.notna(c)]
    if len(cards) > MAX_CLIQUE_SIZE:
        cards = cards[:MAX_CLIQUE_SIZE]
    for i in range(len(cards)):
        for j in range(i+1, len(cards)):
            u, v = cards[i], cards[j]
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

print(f'Graphe : {G.number_of_nodes():,} noeuds (cartes), {G.number_of_edges():,} aretes')
print(f'Densite : {nx.density(G):.6f}')
print(f'Composantes connexes : {nx.number_connected_components(G):,}')

### 2.1 Visualisation d'un cluster suspect

In [ ]:
largest_cc = max(nx.connected_components(G), key=len)
G_sub = G.subgraph(largest_cc).copy()
print(f'Plus grande composante : {G_sub.number_of_nodes():,} noeuds, {G_sub.number_of_edges():,} aretes')

# Sous-graphe suspect : nœuds avec degree >= percentile 90
degrees = dict(G_sub.degree())
threshold = np.percentile(list(degrees.values()), 90)
suspect_nodes = [n for n, d in degrees.items() if d >= threshold]
G_suspect = G_sub.subgraph(suspect_nodes).copy()

fraud_nodes  = [n for n in G_suspect.nodes() if G_suspect.nodes[n].get('is_fraud', 0) == 1]
normal_nodes = [n for n in G_suspect.nodes() if G_suspect.nodes[n].get('is_fraud', 0) == 0]
print(f'Cluster suspect : {G_suspect.number_of_nodes()} noeuds ({len(fraud_nodes)} frauduleux)')

fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(G_suspect, seed=RANDOM_STATE)
nx.draw_networkx_nodes(G_suspect, pos, nodelist=normal_nodes, node_color='steelblue',
                       node_size=50, alpha=0.6, ax=ax)
nx.draw_networkx_nodes(G_suspect, pos, nodelist=fraud_nodes, node_color='crimson',
                       node_size=120, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G_suspect, pos, alpha=0.2, ax=ax)
ax.set_title('Cluster suspect — rouge = frauduleux', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'graph_cluster_suspect.png', dpi=120)
plt.show()

## 3. Features graphe sur le dataset complet

On calcule maintenant les features de structure graphe sur l'ensemble des 590k transactions :
- **`graph_distinct_merchants_7d`** : vélocité inter-marchands sur 7 jours glissants
- **`graph_degree`** : degré du nœud dans G_full (nombre de cartes voisines)
- **`graph_betweenness`** : centralité betweenness (approximée, k=500)

In [ ]:
print('Calcul des features graphe...')
print('Calcul velocite inter-marchands (rolling 7j)...')

# ── Vélocité inter-marchands : sliding window O(n) par groupe ────────────────
# Algorithme deux-pointeurs (left/right) sur le groupe card1 trié par jour.
# Complexité : O(n log n) globale (dominée par le sort), O(k) mémoire par groupe
# (k = nb transactions dans le groupe, jamais le produit cartésien).
# Aucun merge cross-join → pas de MemoryError.

df_vel = (
    df[['card1', 'day', 'P_emaildomain']]
    .dropna(subset=['P_emaildomain'])
    .sort_values(['card1', 'day'])
    .reset_index(drop=True)
)

def _rolling_distinct_7d(group: pd.DataFrame) -> pd.Series:
    """Nombre de P_emaildomain distincts dans les 7 jours précédant chaque transaction.
    Implémentation O(n) par groupe via deux pointeurs (sliding window).
    Le groupe doit être trié par 'day' (garanti par le sort_values ci-dessus).
    """
    days    = group['day'].values
    domains = group['P_emaildomain'].values
    n       = len(days)
    result  = np.empty(n, dtype=np.float32)
    left    = 0
    window: dict = {}

    for right in range(n):
        d = str(domains[right])
        window[d] = window.get(d, 0) + 1
        # Expirer les entrées hors fenêtre (> 7 jours)
        while days[right] - days[left] > 7:
            old = str(domains[left])
            window[old] -= 1
            if window[old] == 0:
                del window[old]
            left += 1
        result[right] = float(len(window))

    return pd.Series(result, index=group.index)


df['graph_distinct_merchants_7d'] = (
    df_vel
    .groupby('card1', sort=False, group_keys=False)
    .apply(_rolling_distinct_7d)
    .reindex(df.index)
    .fillna(1.0)
    .astype(np.float32)
)

print(f"graph_distinct_merchants_7d — moyenne : {df['graph_distinct_merchants_7d'].mean():.3f}, "
      f"max : {df['graph_distinct_merchants_7d'].max():.0f}")
print('Features graphe OK')

## 4. Construction du graphe complet G_full (IEEE-CIS)

On construit maintenant le graphe sur l'ensemble du dataset pour en extraire les features de centralité.

In [ ]:
print('Construction de G_full...')
G_full = nx.Graph()
G_full.add_nodes_from(df['card1'].dropna().astype(str).unique())

grp_full = df.groupby(['P_emaildomain', 'day'])['card1'].apply(list)
for cards in grp_full:
    cards = [str(c) for c in cards if pd.notna(c)]
    if len(cards) > MAX_CLIQUE_SIZE:      # guard identique à la cellule sample
        cards = cards[:MAX_CLIQUE_SIZE]
    for i in range(len(cards)):
        for j in range(i+1, len(cards)):
            u, v = cards[i], cards[j]
            if G_full.has_edge(u, v):
                G_full[u][v]['weight'] += 1
            else:
                G_full.add_edge(u, v, weight=1)

print(f'G_full : {G_full.number_of_nodes():,} noeuds, {G_full.number_of_edges():,} aretes')
print(f'Densité : {nx.density(G_full):.8f}')

## 5. Features de centralité (G_full)

In [ ]:
print('Calcul des features de centralité (approximation k=500)...')

# Degré
degree_dict = dict(G_full.degree())
df['graph_degree'] = df['card1'].astype(str).map(degree_dict).fillna(0).astype(np.float32)

# Betweenness approximée — k=min(500, n_nodes) pour éviter O(VE) exact
betweenness_dict = nx.betweenness_centrality(
    G_full, normalized=True, weight='weight',
    k=min(500, G_full.number_of_nodes())
)
df['graph_betweenness'] = df['card1'].astype(str).map(betweenness_dict).fillna(0).astype(np.float32)

print(f"graph_degree      — moyenne : {df['graph_degree'].mean():.3f}, max : {df['graph_degree'].max():.0f}")
print(f"graph_betweenness — moyenne : {df['graph_betweenness'].mean():.6f}, max : {df['graph_betweenness'].max():.6f}")
print('Centralité OK')